In [41]:
import numpy as np
import random as rdm

n_delay = 3          #low, medium, high 
n_cause = 5          #mechanical, operational, passenger, traffic, external 
n_peak = 2           #peak or non-peak
n_actions = 3        # do nothing, dispatch backup, short turn

n_states = n_delay * n_cause * n_peak

def encode_state(delay, cause, peak):
    return delay * (n_cause * n_peak) + cause * n_peak + peak

def decode_state(state):
    delay = state // (n_cause * n_peak)
    rem = state % (n_cause * n_peak)
    cause = rem // n_peak 
    peak = rem % n_peak
    return delay, cause, peak

def simulate_transition(delay, cause, peak, action):
    ## transition probabilities
    if action == 0:
        probs = {
            0: [0.7, 0.25, 0.05], 
            1: [0.2, 0.5, 0.3],
            2: [0.1, 0.3, 0.6]
        }
        action_cost = 0
    elif action == 1: ##dispatch backup 
        probs = {
            0: [0.8, 0.18, 0.02], 
            1: [0.4, 0.45, 0.15],
            2: [0.25, 0.5, 0.25]
        }
        action_cost = 0
    else:
        probs = {
            0: [0.75, 0.2, 0.05], 
            1: [0.5, 0.35, 0.15],
            2: [0.35, 0.45, 0.2]
        }
        action_cost = 8

    next_delay = np.random.choice([0, 1, 2], p=probs[delay])

    ##change cause and peak
    next_cause = np.random.choice([0, 1, 2, 3, 4])
    if peak == 0:
        next_peak = np.random.choice([0, 1], p=[0.7, 0.3])
    else:
        next_peak = np.random.choice([0, 1], p=[0.3, 0.7])

    delay_minutes_map = {0: 5, 1: 15, 2: 30}
    next_delay_minutes = delay_minutes_map[next_delay]

    reward = -next_delay_minutes - action_cost

    return next_delay, next_cause, next_peak, reward
    

Lambda=1; m=3;
d=5; s=20;
rho=0.01; c=60;

nweeks = 1000


def Qlearning_transit(m, rho, nweeks=1000):
    gamma = 0.9
    alpha=1
    alpha_min = 0.001
    a_factor = (alpha_min/alpha)**(1/nweeks)     
        # a_factor has to be really close to 1, otherwise after a few weeks, 
        # the learning stops because alpha becomes too small.
        # This formula ensures that at the end of the simulation, alpha is 0.001

    epsilon = 1
    epsilon_min = 0.001
    e_factor = (epsilon_min/epsilon)**(1/nweeks) 
    
    
    Q = np.zeros((n_states, n_actions))

    #randomize initial state
    delay = rdm.randint(0, 2)
    cause = rdm.randint(0, 4)
    peak = rdm.randint(0, 1)
    state = encode_state(delay, cause, peak)
    
    
    for week in range(nweeks):
        
        if rdm.random() < epsilon:
            action = rdm.randint(0, n_actions - 1)
        else:
            action = np.argmax(Q[state, :])
            
        delay, cause, peak = decode_state(state)
        next_delay, next_cause, next_peak, reward = simulate_transition(delay, cause, peak, action)
        next_state = encode_state(next_delay, next_cause, next_peak)
    
        Q[state, action] = (1 - alpha) * Q[state, action] + alpha * (
            reward + gamma * np.max(Q[next_state, :])
        )

        state = next_state
        alpha *= a_factor
        epsilon *= e_factor
    
    
    
    ##p=[]
    ###for i in range(m+1):
        ##p.append(np.argmax(Q[i,:]))

    ##print('m =',m,' rho =',rho,' -> ',p)
    
    ##return p
    policy = np.argmax(Q, axis=1)
    return Q, policy

In [46]:
rho = 0.01
m = 3
for m in range(1,7+1):
    Q, policy = Qlearning_transit(m,rho, nweeks = 1000)



In [54]:
print(Q[10:20:1])

[[-31.60006389 -24.68565194 -26.30334133]
 [-21.196745   -21.18965286 -27.65563257]
 [-19.70284944 -12.6125864  -30.9141469 ]
 [-21.81420614 -15.94307276 -15.95367699]
 [-20.03950647 -17.14040628 -17.15295945]
 [-17.6271043  -14.18955007 -15.05825696]
 [-19.36447515 -19.35920806 -31.9205994 ]
 [-18.90741423 -18.92707442 -19.25966644]
 [-21.84655554 -21.74004837 -21.76615422]
 [-15.33935621 -15.35263686 -15.34182996]]


In [53]:
print(policy)

[1 1 1 0 0 0 1 0 1 0 1 1 1 1 1 1 1 0 1 0 2 1 1 2 0 2 0 2 1 1]


In [47]:
cause_names = ["Traffic", "Mechanical", "Passenger", "Operations", "External"]
action_names = ["Do nothing", "Dispatch backup", "Short turn"]
delay_names = ["Low", "Medium", "High"]

for state in range(len(policy)):
    delay, cause, peak = decode_state(state)
    action = policy[state]

    print(f"Delay={delay_names[delay]}, Cause={cause_names[cause]}, Peak={peak} -> {action_names[action]}")
    

Delay=Low, Cause=Traffic, Peak=0 -> Dispatch backup
Delay=Low, Cause=Traffic, Peak=1 -> Dispatch backup
Delay=Low, Cause=Mechanical, Peak=0 -> Dispatch backup
Delay=Low, Cause=Mechanical, Peak=1 -> Do nothing
Delay=Low, Cause=Passenger, Peak=0 -> Do nothing
Delay=Low, Cause=Passenger, Peak=1 -> Do nothing
Delay=Low, Cause=Operations, Peak=0 -> Dispatch backup
Delay=Low, Cause=Operations, Peak=1 -> Do nothing
Delay=Low, Cause=External, Peak=0 -> Dispatch backup
Delay=Low, Cause=External, Peak=1 -> Do nothing
Delay=Medium, Cause=Traffic, Peak=0 -> Dispatch backup
Delay=Medium, Cause=Traffic, Peak=1 -> Dispatch backup
Delay=Medium, Cause=Mechanical, Peak=0 -> Dispatch backup
Delay=Medium, Cause=Mechanical, Peak=1 -> Dispatch backup
Delay=Medium, Cause=Passenger, Peak=0 -> Dispatch backup
Delay=Medium, Cause=Passenger, Peak=1 -> Dispatch backup
Delay=Medium, Cause=Operations, Peak=0 -> Dispatch backup
Delay=Medium, Cause=Operations, Peak=1 -> Do nothing
Delay=Medium, Cause=External, Peak=0